# CYR-GPU-014 / R1C — Amendment-2 operator launcher v4

Preserves the preregistered R1C scientific protocol and frozen compatibility identity while binding the audited float32 clip-certification repair from RUN_READINESS_V3. Select **T4 GPU** and run Cell 0 → Cell 1. Compatible Drive state is resumed rather than restarted.


In [ ]:
import sys, json, hashlib, subprocess
from pathlib import Path

REPO=Path('/content/An-Ra-the-new-AGI-r1c')
REMOTE='https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git'
BRANCH='cymek-500m-readiness'
ORIGINAL_SCIENCE='2a71cea10ebb7a231834b6b112c49e268e9631a5'
FROZEN_COMPAT='b9e4689bdcfa24a3c7d50b2f337c4e702de0bb8a'
ENGINEERING_FIX='b850861545f79e219b55d6403f81e74f92f1592e'

if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'--single-branch','--depth','300',REMOTE,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'remote','set-url','origin',REMOTE],check=True)
    subprocess.run(['git','-C',str(REPO),'fetch','origin',BRANCH,'--depth','300'],check=True)
# Fetch the exact audited Amendment-2 commit explicitly so an existing shallow
# Colab clone cannot make checkout fail merely because that object is absent.
subprocess.run(['git','-C',str(REPO),'fetch','origin',ENGINEERING_FIX,'--depth','1'],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','-q','--detach',ENGINEERING_FIX],check=True)
assert subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()==ENGINEERING_FIX

ready=json.loads((REPO/'docs/cymek/experiments/CYR-GPU-014-R1C/RUN_READINESS_V3.json').read_text())
assert ready['status']=='R1C_READY_FOR_OPERATOR_CUDA_RUN'
assert ready['scientific_result_status']=='NOT_EXECUTED'
assert ready['ready_for_operator_colab_gpu_run'] is True
assert ready['original_scientific_executable_commit']==ORIGINAL_SCIENCE
assert ready['frozen_compatibility_executable_commit']==FROZEN_COMPAT

expected={
 'anra_v5/cyr_gpu014_r1c_run_v2.py':'1078f0527f9e0c5ba6a94be6659209c0ab71a1cb',
 'anra_v5/cyr_gpu014_r1c_run.py':'5bd880dea143e14348019b9c4f959c760a92f051',
 'v5_training/production_backend.py':'502f6697607af619be8d5c46ce05b5b1501e768b',
 'v5_training/step.py':'0e7983942ff37f84e736746f2724b24bf529fc20'}
for p,sha in expected.items():
    got=subprocess.check_output(['git','-C',str(REPO),'hash-object',p],text=True).strip()
    assert got==sha,(p,got,sha)

pre_text=(REPO/'docs/cymek/experiments/CYR-GPU-014-R1C/PREREGISTRATION.json').read_text()
PRE=Path('/content/CYR_GPU_014_R1C_PREREGISTRATION.json'); PRE.write_text(pre_text)
pre_sha=hashlib.sha256(pre_text.encode()).hexdigest()
subprocess.run([sys.executable,'-m','pip','install','-q','pytest','numpy'],check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_v5_cyr_gpu014_r1c_e2e_preflight.py','tests/test_v5_cyr_gpu014_r1c_compat.py','tests/test_v5_cyr_gpu014_r1c.py','-q'],cwd=REPO,check=True)

import torch
if not torch.cuda.is_available(): raise RuntimeError('Select Runtime -> Change runtime type -> T4 GPU')
print('GPU:',torch.cuda.get_device_name(0),'VRAM GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,2))
from google.colab import drive
drive.mount('/content/drive')
OUT=Path('/content/drive/MyDrive/CYMEK/CYR-GPU-014-R1C'); OUT.mkdir(parents=True,exist_ok=True)
binding_path=OUT/'EXECUTABLE_BINDING.json'; gate=OUT/'PREEXECUTION_GATE.json'
binding={'scientific_executable_commit':FROZEN_COMPAT,'original_scientific_executable_commit':ORIGINAL_SCIENCE,'compatibility_wrapper':'anra_v5/cyr_gpu014_r1c_run_v2.py','preregistration_raw_sha256':pre_sha}
if binding_path.exists():
    assert json.loads(binding_path.read_text())==binding,'Existing R1C binding differs; refusing to rewrite scientific identity'
else: binding_path.write_text(json.dumps(binding,indent=2)+'\n')
amendment={'schema':'anra-cyr-gpu014-r1c-engineering-amendment/v2','scientific_executable_commit':FROZEN_COMPAT,'engineering_fix_commit':ENGINEERING_FIX,'scope':'float32 post-clip certification tolerance only','production_backend_blob_sha':expected['v5_training/production_backend.py'],'step_blob_sha':expected['v5_training/step.py'],'scientific_protocol_changed':False,'readiness_status':ready['status']}
amend_path=OUT/'ENGINEERING_AMENDMENT_2.json'
if amend_path.exists(): assert json.loads(amend_path.read_text())==amendment,'Engineering amendment receipt mismatch'
else: amend_path.write_text(json.dumps(amendment,indent=2)+'\n')

if gate.exists() and json.loads(gate.read_text()).get('status')=='PASS':
    print('Existing R1C PREEXECUTION GATE: PASS — preserving prior gate')
    print('Amendment-2 six-arm regression: PASS at',ENGINEERING_FIX)
else:
    cmd=[sys.executable,'-m','anra_v5.cyr_gpu014_r1c_run_v2','--mode','preflight','--repo',str(REPO),'--out',str(OUT),'--prereg',str(PRE)]
    print('Running R1C CUDA preflight:', ' '.join(cmd),flush=True)
    subprocess.run(cmd,cwd=REPO,check=True)
    assert json.loads(gate.read_text()).get('status')=='PASS'
print('R1C READY | science:',FROZEN_COMPAT,'| amendment-2:',ENGINEERING_FIX)


In [ ]:
cmd=[sys.executable,'-m','anra_v5.cyr_gpu014_r1c_run_v2','--mode','run','--repo',str(REPO),'--out',str(OUT),'--prereg',str(PRE)]
print('Starting/resuming CYR-GPU-014-R1C...',flush=True)
proc=subprocess.run(cmd,cwd=REPO)
print('RETURN CODE:',proc.returncode)
if proc.returncode!=0: raise SystemExit('R1C stopped/failed. Preserve Drive state; inspect receipts before retrying.')
campaign=OUT/'CAMPAIGN_RECEIPT.json'
if campaign.exists():
    body=json.loads(campaign.read_text())
    print('STATUS:',body.get('status'),'COMPLETED:',body.get('completed_arm_count'),'/24','VERDICT:',body.get('decision',{}).get('verdict'))


In [ ]:
from google.colab import files
campaign_path=OUT/'CAMPAIGN_RECEIPT.json'
if not campaign_path.exists(): raise RuntimeError('No CAMPAIGN_RECEIPT.json yet')
campaign=json.loads(campaign_path.read_text()); status=campaign.get('status')
bundle=OUT/('CYMEK_R1C_SOFTMAX_MECHANISM_RESULTS.zip' if status=='COMPLETE' else 'CYMEK_R1C_SOFTMAX_MECHANISM_PARTIAL.zip')
if not bundle.exists(): raise RuntimeError(f'Expected bundle missing: {bundle}')
print('STATUS:',status,'COMPLETED:',campaign.get('completed_arm_count'),'/24')
print('BUNDLE:',bundle.name,'SHA256:',hashlib.sha256(bundle.read_bytes()).hexdigest())
files.download(str(bundle))
